In [ ]:
"""
analyze_backtest_results.py

Utility script to analyse model backtest results and generate visualisations.

Inputs (produced by backtest_all_models.py):
    data/out/results/per_split_metrics.csv
    data/out/results/summary_metrics_by_H.csv

Outputs:
    - Console summary of best models per horizon
    - Console summary of DM-test significance vs Naive0
    - Figures saved under data/out/results/figures/:
        * bar_mae_H{H}.png       (MAE by model for each horizon)
        * bar_da_H{H}.png        (Directional Accuracy by model for each horizon)
        * box_mae_H{H}.png       (per-split MAE distribution by model)
        * box_da_H{H}.png        (per-split DA distribution by model)
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------- Paths ----------------

DIR_RES = Path("data/out/results")
SUMMARY_CSV = DIR_RES / "summary_metrics_by_H.csv"
PER_SPLIT_CSV = DIR_RES / "per_split_metrics.csv"
FIG_DIR = DIR_RES / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


# ---------------- Load data ----------------

def load_results():
    if not SUMMARY_CSV.exists():
        raise FileNotFoundError(f"Summary file not found: {SUMMARY_CSV}")
    if not PER_SPLIT_CSV.exists():
        raise FileNotFoundError(f"Per-split file not found: {PER_SPLIT_CSV}")

    summary = pd.read_csv(SUMMARY_CSV)
    per_split = pd.read_csv(PER_SPLIT_CSV)
    return summary, per_split


# ---------------- Textual summaries ----------------

def print_best_models_per_horizon(summary: pd.DataFrame):
    """
    For each horizon H, print the model with lowest MAE and its main metrics.
    """
    print("\n=== Best model per horizon (by MAE) ===")
    for H in sorted(summary["H"].unique()):
        sub = summary[summary["H"] == H].sort_values("MAE")
        best = sub.iloc[0]
        print(
            f"H={H}h → best: {best['Model']}"
            f" | MAE={best['MAE']:.6f}"
            f" | RMSE={best['RMSE']:.6f}"
            f" | R2={best['R2']:.3f}"
            f" | DA={best['DA']:.3f}"
            f" | sMAPE={best['sMAPE']:.3f}"
            f" | MASE={best['MASE']:.3f}"
            f" | DM_p_vsNaive0={best['DM_p_vsNaive0']:.3f}"
        )


def compute_dm_significance(per_split: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    For each horizon and model, compute how often the model significantly
    beats Naive0 according to DM test (p < alpha).
    """
    rows = []
    for H in sorted(per_split["H"].unique()):
        sub_H = per_split[per_split["H"] == H]
        for model in sorted(sub_H["Model"].unique()):
            if model == "Naive0":
                continue
            sub_m = sub_H[sub_H["Model"] == model]
            n_splits = len(sub_m)
            sig = (sub_m["DM_p_vsNaive0"] < alpha).sum()
            frac_sig = sig / n_splits if n_splits > 0 else np.nan
            rows.append({
                "H": H,
                "Model": model,
                "n_splits": n_splits,
                "n_sig_vsNaive0": sig,
                "frac_sig_vsNaive0": frac_sig
            })
    dm_summary = pd.DataFrame(rows).sort_values(
        ["H", "frac_sig_vsNaive0"], ascending=[True, False]
    )
    return dm_summary


def print_dm_summary(dm_summary: pd.DataFrame, alpha: float = 0.05):
    """
    Print a compact DM significance summary to the console.
    """
    print(f"\n=== DM-test significance vs Naive0 (alpha={alpha:.2f}) ===")
    for H in sorted(dm_summary["H"].unique()):
        sub = dm_summary[dm_summary["H"] == H]
        print(f"\nH={H}h:")
        for _, row in sub.iterrows():
            print(
                f"  {row['Model']:<10} → "
                f"splits={int(row['n_splits'])}, "
                f"significant={int(row['n_sig_vsNaive0'])}, "
                f"fraction={row['frac_sig_vsNaive0']:.2f}"
            )


# ---------------- Visualisations ----------------

def plot_bar_metric_per_horizon(summary: pd.DataFrame,
                                metric: str,
                                ylabel: str = None):
    """
    For each horizon H, create a bar chart of <metric> by model and save to PNG.

    One figure per horizon, no subplots.
    """
    if metric not in summary.columns:
        print(f"[WARN] Metric '{metric}' not in summary; skipping bar plots.")
        return

    for H in sorted(summary["H"].unique()):
        sub = summary[summary["H"] == H].copy()
        sub = sub.sort_values(metric)
        models = sub["Model"].tolist()
        values = sub[metric].values

        plt.figure()
        plt.bar(range(len(models)), values)
        plt.xticks(range(len(models)), models, rotation=45, ha="right")
        plt.title(f"{metric} by model (H={H}h)")
        plt.xlabel("Model")
        plt.ylabel(ylabel if ylabel else metric)
        plt.tight_layout()

        fname = FIG_DIR / f"bar_{metric.lower()}_H{H}.png"
        plt.savefig(fname, dpi=300)
        plt.close()
        print(f"Saved bar plot → {fname}")


def plot_box_metric_per_horizon(per_split: pd.DataFrame,
                                metric: str,
                                ylabel: str = None):
    """
    For each horizon H, create a boxplot of per-split <metric> by model.

    One figure per horizon, no subplots.
    """
    if metric not in per_split.columns:
        print(f"[WARN] Metric '{metric}' not in per_split; skipping box plots.")
        return

    for H in sorted(per_split["H"].unique()):
        sub = per_split[per_split["H"] == H].copy()
        models = sorted(sub["Model"].unique())

        data = []
        labels = []
        for m in models:
            vals = sub[sub["Model"] == m][metric].dropna().values
            if len(vals) == 0:
                continue
            data.append(vals)
            labels.append(m)

        if not data:
            continue

        plt.figure()
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.xticks(rotation=45, ha="right")
        plt.title(f"{metric} distribution across splits (H={H}h)")
        plt.xlabel("Model")
        plt.ylabel(ylabel if ylabel else metric)
        plt.tight_layout()

        fname = FIG_DIR / f"box_{metric.lower()}_H{H}.png"
        plt.savefig(fname, dpi=300)
        plt.close()
        print(f"Saved boxplot → {fname}")


def main():
    summary, per_split = load_results()

    # 1) Console summaries
    print_best_models_per_horizon(summary)

    dm_summary = compute_dm_significance(per_split, alpha=0.05)
    print_dm_summary(dm_summary, alpha=0.05)

    # 2) Visualisations
    # Bar charts based on summary (mean across splits)
    plot_bar_metric_per_horizon(summary, metric="MAE", ylabel="MAE (mean across splits)")
    plot_bar_metric_per_horizon(summary, metric="DA", ylabel="Directional accuracy")

    # Boxplots based on per-split metrics
    plot_box_metric_per_horizon(per_split, metric="MAE", ylabel="MAE per split")
    plot_box_metric_per_horizon(per_split, metric="DA", ylabel="Directional accuracy per split")

    print("\nAnalysis and visualisations completed.")


if __name__ == "__main__":
    main()
